# 01 · ETL e validação de dados

**Objetivo.** Transformar os lotes brutos publicados do dataset *Default of Credit Card Clients* (UCI) em um conjunto processado, validado e versionado. Este notebook também abre o ciclo do pipeline (`cycle_id`) e fixa a `data_version` que os notebooks seguintes herdam.

**Entradas.**
- `config/config.yaml`
- `data/raw/source/default of credit card clients.xls`: arquivo original da UCI, baixado e conferido por SHA-256. Nunca é alterado.
- `data/raw/batches/lote_*.csv`: lotes publicados. Simulam a chegada de dados ao longo do tempo.

**Saídas.**
- `data/processed/dataset_processado.parquet`: dados limpos, com atributos derivados e a coluna `particao`.
- `data/processed/etl_metadata.json` e `reports/evidencias/<cycle_id>/01_etl.json`: relatório de validação.
- `reports/ciclo_atual.json`: identidade do ciclo (`cycle_id`, `data_version`, run pai no MLflow), lida pelos notebooks 02 a 05.
- `data/processed/drift.json` e `reports/evidencias/<cycle_id>/01_drift.json`: relatório de drift (Critério 10), lido pelo notebook 04.
- MLflow: run pai do ciclo e runs filhos `etl` e `drift`.

**Metodologia.** Ingestão → versionamento → validação estrutural → nulos → duplicatas → valores inválidos → outliers → limpeza → engenharia de atributos → partição treino/validação/teste → gravação.

Toda a lógica reutilizável está em `src/ingestao.py` e `src/preprocessamento.py`. Este notebook orquestra e documenta.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config" / "config.yaml").is_file())
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

import mlflow

from src import drift, ingestao, preprocessamento as pp, promocao
from src.utils import cycle_run, load_config, start_cycle

pd.set_option("display.max_columns", 50)
cfg = load_config(ROOT)
TARGET = cfg["data"]["target"]

## 1. Ingestão e versionamento

`ensure_initial_batch` baixa a fonte se necessário, confere o checksum e garante que ao menos o `lote_001` foi publicado. A operação é idempotente.

`start_cycle` também cria no MLflow o run pai do ciclo. Cada notebook seguinte abre um run filho nele.

A `data_version` é um hash SHA-256 sobre o nome e o conteúdo de cada lote publicado. Os mesmos lotes sempre geram a mesma versão. Um lote novo (`python -m src.ingestao --novo-lote`) gera uma versão nova.

In [2]:
batch_files = ingestao.ensure_initial_batch(cfg, ROOT)
data_version = ingestao.compute_data_version(batch_files)
ctx = start_cycle(cfg, data_version, [p.name for p in batch_files], ROOT)

print("cycle_id     :", ctx.cycle_id)
print("data_version :", ctx.data_version)
print("lotes        :", ctx.batches)

cycle_id     : ciclo_gha_43986932580_1
data_version : dv_3a043c533b992faa
lotes        : ['lote_001.csv', 'lote_002.csv', 'lote_003.csv']


## 2. Carga dos dados brutos

Na planilha original a primeira linha traz os rótulos `X1..X23, Y`, e o cabeçalho real está na segunda linha. Os lotes em CSV preservam os nomes originais. Aqui o alvo `default payment next month` é renomeado para o nome definido em `data.target`.

In [3]:
raw = pp.rename_target(ingestao.load_released_batches(cfg, ROOT), cfg)
print(raw.shape)
display(raw.head())
display(raw["lote"].value_counts().rename("linhas_por_lote"))

(27000, 26)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month,lote
0,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0,lote_001
1,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0,lote_001
2,6,50000,1,1,2,37,0,0,0,0,0,0,64400,57069,57608,19394,19619,20024,2500,1815,657,1000,1000,800,0,lote_001
3,7,500000,1,1,2,29,0,0,0,0,0,0,367965,412023,445007,542653,483003,473944,55000,40000,38000,20239,13750,13770,0,lote_001
4,8,100000,2,2,2,23,0,-1,-1,0,0,-1,11876,380,601,221,-159,567,380,601,0,581,1687,1542,0,lote_001


lote
lote_001    18000
lote_002     6000
lote_003     3000
Name: linhas_por_lote, dtype: int64

### Dicionário das variáveis (fonte: UCI, Yeh & Lien, 2009)

| Variável | Significado |
|---|---|
| `LIMIT_BAL` | Limite de crédito concedido, em NT$ (dólar taiwanês), incluindo crédito do titular e da família |
| `SEX` | 1 = masculino, 2 = feminino |
| `EDUCATION` | 1 = pós-graduação, 2 = universidade, 3 = ensino médio, 4 = outros |
| `MARRIAGE` | 1 = casado, 2 = solteiro, 3 = outros |
| `AGE` | Idade em anos |
| `PAY_0, PAY_2..PAY_6` | Status de pagamento de set/2005 (`PAY_0`) até abr/2005 (`PAY_6`). -1 = pago em dia; 1..9 = meses de atraso. A nomenclatura original pula `PAY_1` |
| `BILL_AMT1..6` | Valor da fatura de set/2005 a abr/2005 (NT$) |
| `PAY_AMT1..6` | Valor pago de set/2005 a abr/2005 (NT$) |
| `default.payment.next.month` | **Alvo.** 1 = inadimplente no mês seguinte |

## 3. Validação estrutural, nulos, duplicatas e valores inválidos

`pp.validate` confere cada coluna contra a tabela `SCHEMA`, que define o domínio válido. Erros estruturais (coluna ausente, ID duplicado, alvo nulo ou fora de {0, 1}) interrompem o pipeline. Isso impede que um ciclo com dados corrompidos avance até a promoção de modelo.

In [4]:
report = pp.validate(raw, cfg)
if not report.is_valid:
    raise pp.DataValidationError(report.errors)

display(pd.Series({
    "linhas": report.n_rows,
    "colunas": report.n_columns,
    "colunas ausentes": report.missing_columns,
    "colunas inesperadas": report.unexpected_columns,
    "valores nulos": report.missing_values or "nenhum",
    "IDs duplicados": report.duplicate_ids,
    "linhas duplicadas (ignorando ID)": report.duplicate_rows_excluding_id,
    "valores fora do domínio documentado": report.invalid_values,
    "códigos não documentados": report.undocumented_codes,
    "balanço do alvo": report.class_balance,
}, name="valor").to_frame())

,valor
linhas,27000
colunas,26
colunas ausentes,[]
colunas inesperadas,[]
valores nulos,nenhum
IDs duplicados,0
linhas duplicadas (ignorando ID),25
valores fora do domínio documentado,"{'EDUCATION': 308, 'MARRIAGE': 49}"
códigos não documentados,"{'EDUCATION': {5: 247, 6: 48, 0: 13}, 'MARRIAG..."
balanço do alvo,"{0: 0.7792, 1: 0.2208}"


**Leitura dos resultados.**

- **Nulos.** A fonte não tem valores ausentes. O pré-processador ainda inclui `SimpleImputer` para lotes futuros. Ele é ajustado só no treino.
- **Duplicatas.** Não há IDs repetidos. Existem linhas idênticas em todos os atributos e no alvo, mas com IDs diferentes. São clientes distintos com o mesmo perfil, em geral contas sem movimento. Por isso são **mantidas**.
- **Códigos não documentados.** `EDUCATION` ∈ {0, 5, 6} e `MARRIAGE` = 0 não constam no dicionário da UCI. São recodificados para a categoria "outros" (4 e 3). `PAY_x` = -2 e 0 também não constam no dicionário, mas são frequentes e interpretados na literatura como "sem consumo" e "crédito rotativo". Por isso são aceitos.
- **Desbalanceamento.** Cerca de 22% de inadimplentes. A acurácia seria enganosa, porque prever sempre "adimplente" já acerta cerca de 78%. Daí a métrica primária F1.

## 4. Outliers

Critério IQR: valores fora de [Q1 − 1,5·IQR, Q3 + 1,5·IQR]. É apenas diagnóstico.

In [5]:
outliers = pp.outlier_summary(raw)
display(outliers.sort_values("pct_outliers", ascending=False).reset_index(drop=True))

,coluna,q1,q3,limite_inf,limite_sup,min,max,n_outliers,pct_outliers
0,PAY_AMT4,300.00,4040.00,-5310.000,9650.000,0,621000,2681,0.0993
1,PAY_AMT6,121.00,4000.00,-5697.500,9818.500,0,528666,2652,0.0982
2,PAY_AMT5,248.00,4041.25,-5441.875,9731.125,0,426529,2649,0.0981
3,PAY_AMT2,836.00,5000.00,-5410.000,11246.000,0,1684259,2460,0.0911
4,PAY_AMT1,1000.00,5011.00,-5016.500,11027.500,0,873552,2455,0.0909
5,BILL_AMT5,1774.75,50468.50,-71265.875,123509.125,-81334,927171,2444,0.0905
6,BILL_AMT6,1261.00,49406.00,-70956.500,121623.500,-339603,961664,2415,0.0894
7,BILL_AMT4,2300.00,54723.50,-76335.250,133358.750,-170000,891586,2355,0.0872
8,PAY_AMT3,390.00,4503.00,-5779.500,10672.500,0,896040,2332,0.0864
9,BILL_AMT3,2685.50,60311.50,-83753.500,146750.500,-157264,1664089,2225,0.0824


**Decisão.** Os outliers **não são removidos**. Faturas e pagamentos altos são clientes reais de alto limite, e não erros de digitação: todos os valores respeitam o domínio validado na seção 3. Removê-los enviesaria o modelo contra exatamente esse público. A Random Forest é insensível à escala. Para a Regressão Logística, a padronização é ajustada só no treino.

## 5. Limpeza

Aplica a recodificação dos códigos não documentados e remove linhas que continuem inválidas. Se a fração removida passar de `data.validation.max_invalid_row_fraction`, o ETL falha.

In [6]:
clean, cleaning_log = pp.clean(raw, cfg)
display(pd.Series(cleaning_log, name="contagem").to_frame())
display(clean["EDUCATION"].value_counts().sort_index().rename("EDUCATION após limpeza"))

,contagem
recodificados_EDUCATION,308
recodificados_MARRIAGE,49
linhas_removidas,0


EDUCATION
1     9548
2    12621
3     4412
4      419
Name: EDUCATION após limpeza, dtype: int64

## 6. Engenharia de atributos

Cada atributo usa apenas valores da própria linha. Não há estatística agregada do dataset, logo não há vazamento entre treino e teste.

| Atributo | Definição | Justificativa |
|---|---|---|
| `UTILIZACAO_ATUAL` | `BILL_AMT1 / LIMIT_BAL` | Uso do limite no último mês |
| `UTILIZACAO_MEDIA` | média(`BILL_AMT1..6`) / `LIMIT_BAL` | Uso típico do limite no semestre |
| `RAZAO_PAGAMENTO` | Σ`PAY_AMT` / Σ max(`BILL_AMT`, 0), limitado a [0, 5] | Quanto da dívida o cliente costuma quitar |
| `MESES_EM_ATRASO` | nº de meses com `PAY_x` ≥ 1 | Frequência de atraso |
| `ATRASO_MAXIMO` | max(`PAY_x`) | Gravidade do pior atraso |
| `TENDENCIA_FATURA` | (`BILL_AMT1` − `BILL_AMT6`) / `LIMIT_BAL` | Endividamento crescente ou decrescente |

In [7]:
features_df = pp.engineer_features(clean)
display(features_df[pp.ENGINEERED].describe().T)
display(features_df.groupby(TARGET)[pp.ENGINEERED].mean().T.rename(columns={0: "adimplente", 1: "inadimplente"}))

,count,mean,std,min,25%,50%,75%,max
UTILIZACAO_ATUAL,27000.0,0.423900,0.409961,-0.619892,0.021941,0.314359,0.833125,5.3095
UTILIZACAO_MEDIA,27000.0,0.372963,0.349967,-0.232590,0.029928,0.284640,0.687532,2.6738
RAZAO_PAGAMENTO,27000.0,0.405472,0.599203,0.000000,0.043141,0.106365,0.713441,5.0000
MESES_EM_ATRASO,27000.0,0.834444,1.554592,0.000000,0.000000,0.000000,1.000000,6.0000
ATRASO_MAXIMO,27000.0,0.438148,1.343331,-2.000000,0.000000,0.000000,2.000000,8.0000
TENDENCIA_FATURA,27000.0,0.105456,0.301320,-1.827300,-0.028812,0.006944,0.181808,5.3095


default.payment.next.month,adimplente,inadimplente
UTILIZACAO_ATUAL,0.405083,0.490296
UTILIZACAO_MEDIA,0.351670,0.448100
RAZAO_PAGAMENTO,0.427528,0.327644
MESES_EM_ATRASO,0.504421,1.998994
ATRASO_MAXIMO,0.202491,1.269708
TENDENCIA_FATURA,0.108563,0.094495


## 7. Protocolo de avaliação: treino / validação / teste

O protocolo é definido **antes** de qualquer treino.

- **Partição por hash do ID.** Cada cliente recebe um número em [0, 1] derivado de `sha256(random_state:ID)`. Abaixo de `test_size` vai para **teste**. Entre `test_size` e `test_size + validation_size` vai para **validação**. O restante vai para **treino**. Com a configuração atual a divisão é 60/20/20.
- **Por que hash e não `train_test_split`?** Quando um lote novo chega, um sorteio aleatório reembaralharia tudo, e clientes do teste do ciclo 1 poderiam virar treino no ciclo 2. Com o hash, cada cliente fica sempre na mesma partição. O teste continua isolado entre ciclos.
- **Uso de cada partição.** O treino ajusta o pré-processador, os seletores de atributos e os modelos. A validação decide a competição entre modelos e a promoção. O teste é usado **só** na avaliação final, nunca para decisões recorrentes.
- O pré-processamento com parâmetros (imputação, one-hot, padronização) fica em `pp.build_preprocessor` e é ajustado apenas no treino.

In [8]:
dataset = features_df.assign(particao=pp.assign_split(features_df[cfg["data"]["id_column"]], cfg))

split_table = dataset.groupby("particao").agg(linhas=(TARGET, "size"), taxa_inadimplencia=(TARGET, "mean"))
split_table["fracao"] = split_table["linhas"] / len(dataset)
display(split_table.round(4))

assert pp.assign_split(dataset[cfg["data"]["id_column"]], cfg).equals(dataset["particao"]), "particao deterministica"
assert not set(dataset.loc[dataset.particao == "teste", "ID"]) & set(dataset.loc[dataset.particao != "teste", "ID"]), "teste isolado"

,linhas,taxa_inadimplencia,fracao
particao,,,
teste,5322,0.2129,0.1971
treino,16239,0.2191,0.6014
validacao,5439,0.2337,0.2014


## 8. Separação X / y e verificação do pré-processador

Separa atributos e alvo e ajusta o `ColumnTransformer` **somente no treino**. Validação e teste apenas recebem `transform`. Esta célula comprova que o pré-processador funciona. O ajuste definitivo acontece dentro do pipeline de cada modelo, no notebook 03.

In [9]:
features = pp.feature_columns(cfg, dataset)
train = dataset[dataset.particao == "treino"]
X_train, y_train = train[features], train[TARGET]

preprocessor = pp.build_preprocessor(features).fit(X_train)
X_valid = preprocessor.transform(dataset.loc[dataset.particao == "validacao", features])

print("atributos candidatos :", len(features))
print("após one-hot         :", len(preprocessor.get_feature_names_out()))
print("X_train              :", X_train.shape, "| X_valid transformado:", X_valid.shape)

atributos candidatos : 29
após one-hot         : 35
X_train              : (16239, 29) | X_valid transformado: (5439, 35)


## 9. Gravação do dataset processado

In [10]:
path = pp.save_processed(dataset, ctx, cfg, {
    "validacao": report.to_dict(),
    "limpeza": cleaning_log,
    "outliers": outliers.set_index("coluna")["n_outliers"].to_dict(),
    "atributos_derivados": pp.ENGINEERED,
    "atributos_candidatos": features,
}, ROOT)
print(path.relative_to(ROOT))

2026-11-17 00:22:35,758 INFO src.preprocessamento: Dataset processado salvo: dataset_processado.parquet (27000 linhas, dv_3a043c533b992faa)


data/processed/dataset_processado.parquet


## 10. Registro no MLflow

O run filho `etl` guarda o tamanho dos dados, a taxa de inadimplência por partição e o relatório de validação completo.

In [11]:
with cycle_run(ctx, cfg, stage="etl", root=ROOT):
    mlflow.log_params({"n_linhas": len(dataset), "n_lotes": len(ctx.batches), "n_atributos_candidatos": len(features),
                       "test_size": cfg["evaluation"]["test_size"], "validation_size": cfg["evaluation"]["validation_size"],
                       "random_state": cfg["data"]["random_state"]})
    mlflow.log_metrics({f"linhas_{k}": int(v) for k, v in split_table["linhas"].items()})
    mlflow.log_metrics({f"taxa_inadimplencia_{k}": float(v) for k, v in split_table["taxa_inadimplencia"].items()})
    mlflow.log_dict(report.to_dict(), "validacao.json")
    mlflow.log_dict(cleaning_log, "limpeza.json")
print("run etl registrado no ciclo", ctx.cycle_id)

run etl registrado no ciclo ciclo_gha_43986932580_1


## 11. Monitoramento de data drift (Critério 10)

**Problema.** Um modelo de crédito aprende a relação entre perfil e inadimplência em uma época específica. Se a carteira muda (novos limites, outro público, crise econômica), as entradas deixam de parecer com o treino e o desempenho cai sem aviso. Isso é *data drift*.

**Método.** *Population Stability Index* (PSI) por atributo, entre:
- **referência**: os lotes com que o modelo vigente foi treinado, registrados no manifesto `models/modelo_vigente.json`;
- **atual**: os lotes publicados depois disso.

`PSI = Σ (p_atual − p_ref) · ln(p_atual / p_ref)`. Atributos contínuos usam 10 faixas por quantis da referência. Atributos discretos usam um valor por faixa. Os limiares em `drift.thresholds` seguem a convenção do setor de crédito: abaixo de 0,1 é **estável**, de 0,1 a 0,25 é **moderado**, a partir de 0,25 é **severo**.

**Efeito no pipeline.** O status do ciclo é o pior status entre os atributos que o modelo vigente consome. Com drift **severo** e `drift.relax_promotion_on_severe: true`, o notebook 04 zera a melhoria mínima exigida: um desafiante treinado com os dados novos substitui o vigente desde que não seja pior. Sem modelo vigente (primeiro ciclo), não há referência e o drift não é calculado.

In [12]:
manifest = promocao.load_manifest(cfg, ROOT)
drift_result = drift.analyze_new_batches(dataset, manifest, features, cfg)
drift.save_drift(drift_result, ctx, cfg, ROOT)
drift.log_drift(drift_result, ctx, cfg, ROOT)

print("status do drift :", drift_result.status)
print("referência      :", drift_result.reference_batches or "-")
print("lotes novos     :", drift_result.current_batches)
if not drift_result.table.empty:
    display(drift_result.table.assign(monitorado=drift_result.table["feature"].isin(drift_result.monitored_features)))

status do drift : estavel
referência      : ['lote_001']
lotes novos     : ['lote_002', 'lote_003']


,feature,psi,status,monitorado
0,AGE,3.538923e-03,estavel,False
1,BILL_AMT4,2.493245e-03,estavel,False
2,BILL_AMT3,2.454168e-03,estavel,False
3,PAY_AMT1,2.267731e-03,estavel,True
4,BILL_AMT2,2.212947e-03,estavel,False
5,TENDENCIA_FATURA,1.763192e-03,estavel,False
6,PAY_AMT5,1.456786e-03,estavel,False
7,BILL_AMT5,1.382909e-03,estavel,False
8,PAY_6,1.378235e-03,estavel,False
9,PAY_AMT2,1.321100e-03,estavel,False


### Controle positivo do detector

Os lotes deste projeto são partições aleatórias da mesma fonte, então o esperado é *estável*. Para provar que o detector reconhece um drift real, a célula abaixo aplica um deslocamento **sintético e conhecido** a uma cópia da partição de validação: limites 40% menores, um mês a mais de atraso em `PAY_0` e clientes 8 anos mais velhos. Essa cópia é comparada com o treino.

Esta célula é apenas evidência. Ela não grava nada e não influencia nenhuma decisão do pipeline.

In [13]:
reference = dataset[dataset.particao == "treino"]
untouched = dataset[dataset.particao == "validacao"]
shifted = untouched.assign(LIMIT_BAL=untouched.LIMIT_BAL * 0.6, PAY_0=(untouched.PAY_0 + 1).clip(upper=8), AGE=untouched.AGE + 8)
shifted = pp.engineer_features(shifted)

control = (drift.drift_table(reference, untouched, ["LIMIT_BAL", "PAY_0", "AGE", "UTILIZACAO_ATUAL", "BILL_AMT1"], cfg)
           .merge(drift.drift_table(reference, shifted, ["LIMIT_BAL", "PAY_0", "AGE", "UTILIZACAO_ATUAL", "BILL_AMT1"], cfg),
                  on="feature", suffixes=("_sem_deslocamento", "_com_deslocamento")))
display(control)
assert (control["status_sem_deslocamento"] == "estavel").all(), "amostras da mesma populacao devem ser estaveis"
assert (control.set_index("feature").loc[["LIMIT_BAL", "PAY_0", "AGE"], "status_com_deslocamento"] == "severo").all(), "deslocamento conhecido deve ser severo"

,feature,psi_sem_deslocamento,status_sem_deslocamento,psi_com_deslocamento,status_com_deslocamento
0,AGE,0.002922,estavel,2.249428,severo
1,UTILIZACAO_ATUAL,0.001870,estavel,0.535278,severo
2,PAY_0,0.001017,estavel,1.570642,severo
3,LIMIT_BAL,0.000930,estavel,0.764869,severo
4,BILL_AMT1,0.000872,estavel,0.000872,estavel


## Conclusões

- O ciclo foi aberto com um `cycle_id` e uma `data_version` derivada do conteúdo dos lotes. Os dois seguem para os notebooks 02 a 05 via `reports/ciclo_atual.json` e estão em todos os runs do MLflow.
- A validação bloqueia o pipeline em caso de erro estrutural. Códigos não documentados são recodificados de forma explícita e registrada.
- Outliers são reais e foram mantidos. Duplicatas por perfil são clientes distintos e também foram mantidas.
- A partição por hash garante que o conjunto de teste continue isolado quando novos lotes chegam.
- Nenhuma transformação com parâmetros foi ajustada fora do treino.
- O relatório de drift compara os dados do modelo vigente com os lotes novos e informa a decisão de promoção. O controle positivo mostra que o PSI separa uma amostra da mesma população de uma amostra deslocada.

**Próximo passo:** `02_votacao_features.ipynb`, com três métodos de seleção de atributos e votação.